# Introduction of Backtest — ห้องทดลองออฟไลน์

ตัวอย่างนี้ใช้ **ข้อมูลสมมติทั้งหมด** ไม่มีราคาตลาด ไม่มีวันที่จริง และไม่มีการดาวน์โหลดข้อมูล
กำหนด seed **20260925** ก่อนประเมินผล และใช้หนึ่งเส้นทางเพื่อสอนกลไก ไม่ได้ค้นหา seed หรือพารามิเตอร์ที่กำไรดี

กฎคงที่: SMA 20/50 ของ close เป็น long/cash; ประเมิน sample volatility จาก 20 close-to-close simple returns
และ annualize ด้วย √252; target volatility 10% ต่อปี; จำกัดน้ำหนักไม่เกิน 1; เงินสดและ risk-free rate 0%
ซื้อขายได้เป็นเศษหุ้น ไม่มี leverage, ดอกเบี้ย, ภาษี หรือข้อจำกัดปริมาณซื้อขาย

แบบใช้ข้อมูลทันเวลา: ใช้ close ของ session t−1 ตัดสินใจ แล้วซื้อขายที่ open t และถือจน open t+1
แบบผิด: แอบใช้ close t และ volatility ที่รวม close t ไปซื้อขายที่ open t ซึ่งยังไม่รู้ข้อมูลนั้น
ทุกแบบประเมินช่วง open60 → open319 เท่ากัน รวม 259 ช่วงผลตอบแทน

Notebook นี้ใช้เพียง Python standard library รันทุก cell จากบนลงล่างได้โดยไม่ต้องติดตั้งแพ็กเกจ


In [1]:
"""Offline teaching backtest: invented prices, fixed seed, no market data.

Run with Python 3 using only its standard library. A session number is an
observation index, not a date. All prices are in invented currency units.
"""

import json
import math
import random
import statistics

SEED = 20260925
SESSION_COUNT = 320
START_SESSION = 60
PERIODS_PER_YEAR = 252


def make_data(seed=SEED, sessions=SESSION_COUNT):
    """Generate one fixed path; the seed was fixed before evaluating results."""
    rng = random.Random(seed)
    rows = []
    previous_close = 100.0
    for session in range(sessions):
        opening = previous_close * math.exp(0.0001 + 0.004 * rng.gauss(0, 1))
        intraday_sigma = 0.009 + 0.004 * math.sin(2 * math.pi * session / 80)
        closing = opening * math.exp(0.0002 + intraday_sigma * rng.gauss(0, 1))
        rows.append({"session": session, "open": round(opening, 8),
                     "close": round(closing, 8)})
        previous_close = closing
    return {"metadata": {
        "seed": seed, "sessions": sessions, "kind": "synthetic teaching data",
        "units": "invented currency units per share; session indices, no calendar dates",
        "generator": "Python random.Random(seed), Gaussian log-price increments",
        "openRule": "previous_close * exp(0.0001 + 0.004 * Z_open)",
        "closeRule": "open * exp(0.0002 + (0.009 + 0.004*sin(2*pi*session/80)) * Z_close)",
        "normalDraws": "Independent standard Normal draws; first previous_close = 100",
        "rounding": "Published open and close prices rounded to 8 decimal places",
        "purpose": "Explain timing, costs, and sizing; not evidence of a profitable strategy",
    }, "rows": rows}


def sample_std(values):
    return statistics.stdev(values) if len(values) > 1 else 0.0


def indicators(rows, index):
    closes = [row["close"] for row in rows]
    fast = sum(closes[index - 19:index + 1]) / 20
    slow = sum(closes[index - 49:index + 1]) / 50
    returns = [closes[k] / closes[k - 1] - 1 for k in range(index - 19, index + 1)]
    return {"side": 1.0 if fast > slow else 0.0, "fast": fast, "slow": slow,
            "volatility": sample_std(returns) * math.sqrt(PERIODS_PER_YEAR)}


def rebalance(pretrade_weight, target_weight, cost_fraction):
    """Exact post-fee target weight, with proportional cost on traded notional.

    E is pretrade equity, a = risky holdings / E, w = target AFTER costs.
    k = E_after_cost / E solves k = 1 - c * abs(w*k - a).
    No fixed fees, spreads beyond the supplied cost, interest, or shorting.
    """
    a, w, c = pretrade_weight, target_weight, cost_fraction
    if w >= a:
        factor = (1 + c * a) / (1 + c * w)
    else:
        factor = (1 - c * a) / (1 - c * w)
    turnover = abs(w * factor - a)
    return {"factor": factor, "turnover": turnover, "feeFraction": c * turnover}


def evaluate_backtest(data, lookahead=False, vol_target=False, cost_bps=5,
                      target_vol=0.1, benchmark=False):
    rows = data["rows"] if isinstance(data, dict) else data
    if len(rows) <= START_SESSION + 1:
        raise ValueError("At least 62 price rows are required")
    if not 0 <= cost_bps < 10000 or not math.isfinite(cost_bps):
        raise ValueError("cost_bps must be finite and between 0 and 10000 (exclusive)")
    if not target_vol > 0 or not math.isfinite(target_vol):
        raise ValueError("target_vol must be finite and positive")
    if any(not math.isfinite(row[key]) or row[key] <= 0
           for row in rows for key in ("open", "close")):
        raise ValueError("All open and close prices must be finite and positive")
    c = cost_bps / 10000
    equity, returns, weights, audit = [1.0], [], [], []
    sessions = [rows[START_SESSION]["session"]]
    pretrade_weight, total_turnover = 0.0, 0.0
    for t in range(START_SESSION, len(rows) - 1):
        decision_index = t if lookahead else t - 1
        info = indicators(rows, decision_index)
        size = min(1.0, target_vol / info["volatility"]) if info["volatility"] > 0 else 0.0
        weight = 1.0 if benchmark else info["side"] * (size if vol_target else 1.0)
        trade = rebalance(pretrade_weight, weight, c)
        asset_return = rows[t + 1]["open"] / rows[t]["open"] - 1
        holding_factor = 1 + weight * asset_return
        period_factor = trade["factor"] * holding_factor
        drifted_weight = weight * (1 + asset_return) / holding_factor
        final_turnover = drifted_weight if t == len(rows) - 2 else 0.0
        if final_turnover:
            period_factor *= 1 - c * final_turnover
        total_turnover += trade["turnover"] + final_turnover
        returns.append(period_factor - 1)
        equity.append(equity[-1] * period_factor)
        sessions.append(rows[t + 1]["session"])
        weights.append(weight)
        audit.append({"session": rows[t]["session"], "decisionSession": rows[decision_index]["session"],
                      "pretradeWeight": pretrade_weight, "weight": weight,
                      "estimatedVolatility": info["volatility"], "side": info["side"],
                      "assetReturn": asset_return, "rebalanceTurnover": trade["turnover"],
                      "rebalanceFeeFraction": trade["feeFraction"],
                      "postFeeEquityFactor": trade["factor"],
                      "finalLiquidationTurnover": final_turnover})
        pretrade_weight = drifted_weight
    peak, drawdowns = 1.0, []
    for value in equity:
        peak = max(peak, value)
        drawdowns.append(value / peak - 1)
    n = len(returns)
    volatility = sample_std(returns)
    identifier = "benchmark" if benchmark else ("biased" if lookahead else "causal") + ("-target" if vol_target else "-fixed")
    labels = {"causal-fixed": "ใช้ข้อมูลทันเวลา · ขนาดคงที่", "causal-target": "ใช้ข้อมูลทันเวลา · Vol target",
              "biased-fixed": "ใช้ข้อมูลอนาคต · ขนาดคงที่", "biased-target": "ใช้ข้อมูลอนาคต · Vol target",
              "benchmark": "Buy and hold · ช่วงเวลาเดียวกัน"}
    return {"id": identifier, "label": labels[identifier], "returns": returns, "equity": equity,
            "drawdowns": drawdowns, "weights": weights, "sessions": sessions, "audit": audit,
            "metrics": {"totalReturn": equity[-1] - 1, "cagr": equity[-1] ** (PERIODS_PER_YEAR / n) - 1,
                        "volatility": volatility * math.sqrt(PERIODS_PER_YEAR),
                        "sharpe": statistics.mean(returns) / volatility * math.sqrt(PERIODS_PER_YEAR) if volatility else None,
                        "maxDrawdown": min(drawdowns), "turnover": total_turnover},
            "assumptions": {"startSession": rows[START_SESSION]["session"], "endSession": rows[-1]["session"],
                            "periods": n, "periodsPerYear": PERIODS_PER_YEAR, "costBps": cost_bps,
                            "targetVol": target_vol, "leverageCap": 1, "riskFreeRate": 0,
                            "smaWindows": [20, 50], "volatilityWindow": 20,
                            "volatilityEstimator": "sample standard deviation of close-to-close simple returns",
                            "execution": "rebalance at open t, hold to open t+1",
                            "information": "close t (impossible at open t)" if lookahead and not benchmark else "close t-1",
                            "zeroEstimatedVolatility": "hold cash when volatility targeting",
                            "costConvention": "exact target weight after proportional fees; initial entry and final liquidation included",
                            "turnoverConvention": "sum absolute traded notional / equity immediately before each trade event"}}


def compare_backtests(data, cost_bps=5, target_vol=0.1):
    return [evaluate_backtest(data, lookahead=lookahead, vol_target=vol_target,
                              cost_bps=cost_bps, target_vol=target_vol)
            for lookahead, vol_target in [(False, False), (False, True), (True, False), (True, True)]]


def evaluate_benchmark(data, cost_bps=5, target_vol=0.1):
    return evaluate_backtest(data, cost_bps=cost_bps, target_vol=target_vol, benchmark=True)


def print_summary(results):
    print(f"{'scenario':<16} {'total':>10} {'CAGR*':>10} {'vol*':>10} {'Sharpe*':>9} {'MDD':>10} {'turnover':>10}")
    for result in results:
        m = result["metrics"]
        sharpe = f"{m['sharpe']:.3f}" if m["sharpe"] is not None else "undefined"
        print(f"{result['id']:<16} {m['totalReturn']:>9.2%} {m['cagr']:>9.2%} {m['volatility']:>9.2%} {sharpe:>9} {m['maxDrawdown']:>9.2%} {m['turnover']:>10.3f}")
    print("* Annualized using 252 sessions/year; invented sessions, not actual dates. Sharpe uses rf=0.")



print("Definitions loaded: fixed SMA 20/50, volatility window 20, cap 1, offline only.")


Definitions loaded: fixed SMA 20/50, volatility window 20, cap 1, offline only.


## 1. ตรวจข้อมูลและหน่วยก่อนเริ่ม

ราคาเป็นหน่วยเงินสมมติต่อหุ้น; `session` เป็นเลขลำดับ ไม่ใช่วันทำการจริง
252 sessions/year เป็น convention สำหรับแสดง annualized statistics ในตัวอย่างเท่านั้น


In [2]:
data = make_data()
assert len(data["rows"]) == 320
assert data == make_data()
print("Seed:", data["metadata"]["seed"])
print("Rows:", len(data["rows"]))
print("Units:", data["metadata"]["units"])
for row in data["rows"][:3] + data["rows"][-2:]:
    print(row)


Seed: 20260925
Rows: 320
Units: invented currency units per share; session indices, no calendar dates
{'session': 0, 'open': 99.90903769, 'close': 100.34657149}
{'session': 1, 'open': 99.93163627, 'close': 99.8883627}
{'session': 2, 'open': 100.57375504, 'close': 100.02428606}
{'session': 318, 'open': 126.5970698, 'close': 126.30653218}
{'session': 319, 'open': 126.14726615, 'close': 127.1281217}


## 2. เปรียบเทียบสี่กรณีด้วยกฎเดิม

เปลี่ยนเฉพาะความถูกต้องของเวลาและการกำหนดขนาด position
ผลจากกรณีใช้ข้อมูลอนาคตใช้ประเมินการซื้อขายจริงไม่ได้ ไม่ว่าจะสูงหรือต่ำกว่า
Volatility targeting เป็นการเปลี่ยน exposure ไม่ใช่หลักประกันว่าจะเพิ่มผลตอบแทนหรือจำกัด drawdown


In [3]:
results = compare_backtests(data, cost_bps=5, target_vol=0.1)
assert all(len(result["returns"]) == 259 for result in results)
assert all(result["sessions"] == list(range(60, 320)) for result in results)
print_summary(results)


scenario              total      CAGR*       vol*   Sharpe*        MDD   turnover
causal-fixed        13.17%    12.79%    14.49%     0.903    -8.79%      5.999
causal-target        8.76%     8.52%     9.60%     0.899    -6.06%      9.566
biased-fixed        12.61%    12.25%    14.53%     0.868    -8.79%      5.999
biased-target        7.77%     7.55%     9.03%     0.851    -5.29%      9.589
* Annualized using 252 sessions/year; invented sessions, not actual dates. Sharpe uses rf=0.


## 3. ใช้ benchmark ในช่วงเวลาเดียวกัน

Buy and hold ซื้อที่ open60 ถือจน open319 และเสียต้นทุนเข้า/ออก 5 bps แบบเดียวกัน
ไม่มีค่าใช้จ่ายซ้ำทุก session เมื่อถือหุ้นเต็มพอร์ตอยู่แล้ว


In [4]:
benchmark = evaluate_benchmark(data, cost_bps=5)
print_summary([benchmark])
price_ratio = data["rows"][-1]["open"] / data["rows"][60]["open"]
c = 5 / 10000
expected_final_equity = price_ratio * (1 - c) / (1 + c)
assert math.isclose(benchmark["equity"][-1], expected_final_equity, rel_tol=1e-12)
print("Verified buy-and-hold entry/exit cost identity:", round(expected_final_equity, 8))


scenario              total      CAGR*       vol*   Sharpe*        MDD   turnover
benchmark           15.56%    15.11%    16.58%     0.931   -10.73%      2.000
* Annualized using 252 sessions/year; invented sessions, not actual dates. Sharpe uses rf=0.
Verified buy-and-hold entry/exit cost identity: 1.15557695


## 4. ตรวจว่าสัญญาณรู้ได้เมื่อใด

กรณีที่ถูกต้องต้องมี `decisionSession < session` เสมอ
การ lag เพียงหนึ่งแถวไม่ใช่ใบรับรองทั่วไป: ในตัวอย่างนี้เราเลือก next open เป็นราคาซื้อขายที่เกิดหลัง close ที่ใช้ตัดสินใจ


In [5]:
for result in results:
    causal = result["id"].startswith("causal")
    assert all(row["decisionSession"] == row["session"] - int(causal) for row in result["audit"])
    first = result["audit"][0]
    print(result["id"], "execution open", first["session"], "uses close", first["decisionSession"])

# Altering a not-yet-known close must not change the first causal weight.
changed = json.loads(json.dumps(data))
changed["rows"][60]["close"] *= 2
before = evaluate_backtest(data, vol_target=True)["weights"][0]
after = evaluate_backtest(changed, vol_target=True)["weights"][0]
assert before == after
print("Future-close perturbation leaves the first causal weight unchanged:", before)


causal-fixed execution open 60 uses close 59
causal-target execution open 60 uses close 59
biased-fixed execution open 60 uses close 60
biased-target execution open 60 uses close 60
Future-close perturbation leaves the first causal weight unchanged: 0.7058197644116662


## 5. ตรวจบัญชีต้นทุนและการ drift ของน้ำหนัก

ให้ E เป็น equity ก่อนซื้อขาย, a เป็น risky weight ก่อนซื้อขาย, w เป็น risky weight เป้าหมายหลังหักค่าธรรมเนียม
และ c เป็นค่าธรรมเนียมต่อมูลค่าซื้อขาย แล้ว k = equity หลังค่าธรรมเนียม / E แก้สมการ
`k = 1 − c × |w × k − a|` ได้ตรงตัว

หลังถือครองหนึ่งช่วงที่สินทรัพย์มี simple return R:
`portfolio factor = k × (1 + w × R)` และน้ำหนักก่อน rebalance ครั้งถัดไปคือ
`a_next = w × (1 + R) / (1 + w × R)`

การปิดสถานะสุดท้ายที่ open319 หัก `c × risky holdings` และรวมในผลตอบแทนช่วงสุดท้าย
turnover คือผลรวมมูลค่าซื้อขายสัมบูรณ์หาร equity ก่อนแต่ละเหตุการณ์ซื้อขาย จึงไม่ใช่จำนวน trades


In [6]:
trade = rebalance(0.5, 0.8, 0.001)
assert math.isclose(trade["factor"] + trade["feeFraction"], 1.0, abs_tol=1e-12)
assert math.isclose(trade["turnover"], abs(0.8 * trade["factor"] - 0.5), abs_tol=1e-12)
print("Rebalance a=0.5 -> post-fee w=0.8; c=0.1%:", trade)
for result in results + [benchmark]:
    assert all(0 <= weight <= 1 for weight in result["weights"])
    assert math.isclose(math.prod(1 + r for r in result["returns"]), result["equity"][-1], rel_tol=1e-12)
    print(result["id"], "final liquidation turnover:", round(result["audit"][-1]["finalLiquidationTurnover"], 6))
print("Verified: all weights in [0,1], compounded returns equal final equity, entry and final liquidation included.")


Rebalance a=0.5 -> post-fee w=0.8; c=0.1%: {'factor': 0.9997002398081535, 'turnover': 0.2997601918465228, 'feeFraction': 0.0002997601918465228}
causal-fixed final liquidation turnover: 0.0
causal-target final liquidation turnover: 0.0
biased-fixed final liquidation turnover: 0.0
biased-target final liquidation turnover: 0.0
benchmark final liquidation turnover: 1.0
Verified: all weights in [0,1], compounded returns equal final equity, entry and final liquidation included.


## 6. จดหมายตอนเย็น ใช้ตัดสินใจตอนเช้าไม่ได้

ตัวอย่างย่อใช้กฎราคาปิดล่าสุด >100 ให้ Long มิฉะนั้น Cash แยกจาก SMA ในห้องทดลองหลัก
รู้ close0=99 ก่อน open1=100; ต่อมา close1=108 จึงเกิดสัญญาณ Long สำหรับ open2=110
สถานะที่ถือระหว่าง open1 กับ open2 ยังเป็น Cash ผลตอบแทนหุ้นของพอร์ตจึงเป็นศูนย์


In [7]:
close0, open1, close1, open2 = 99.0, 100.0, 108.0, 110.0
causal_weight = float(close0 > 100)
future_signal = float(close1 > 100)
asset_return = open2 / open1 - 1
actual_return = causal_weight * asset_return
invalid_hindsight_return = future_signal * asset_return
assert actual_return == 0
assert math.isclose(invalid_hindsight_return, 0.1)
print("Actual cash return:", f"{actual_return:.0%}")
print("Impossible hindsight claim:", f"{invalid_hindsight_return:.0%}")
print("The close1 signal can affect the NEXT holding interval, not the completed one.")


Actual cash return: 0%
Impossible hindsight claim: 10%
The close1 signal can affect the NEXT holding interval, not the completed one.


## 7. ลมแรงขึ้น จึงลดของบนรถ

ใช้เป้าความผันผวน10% และเพดานน้ำหนัก1 คำนวณ size=min(1,target/estimated_vol)
ตัวเลข weight × estimated_vol เป็นเพียงค่าประมาณตามสัดส่วนภายใต้สมมติฐานของแบบฝึกหัด
ไม่รับประกัน realized volatility หรือ drawdown


In [8]:
target = 0.10
for estimated_vol in [0.05, 0.20, 0.40]:
    weight = min(1.0, target / estimated_vol)
    cash = 1.0 - weight
    scaled_risk = weight * estimated_vol
    assert 0 <= weight <= 1
    print(f"Asset vol {estimated_vol:.0%}: stock {weight:.0%}, cash {cash:.0%}, scaled estimate {scaled_risk:.0%}")
assert math.isclose(min(1.0, 0.1 / 0.2), 0.5)


Asset vol 5%: stock 100%, cash 0%, scaled estimate 5%
Asset vol 20%: stock 50%, cash 50%, scaled estimate 10%
Asset vol 40%: stock 25%, cash 75%, scaled estimate 10%


## 8. ขาดทุนกับการฟื้นตัวใช้ฐานคนละก้อน

จาก100ลง80คือ−20% แต่+20%จาก80ได้เพียง96 ต้อง+25%จึงกลับ100
สูตรสำหรับสัดส่วนขาดทุน d คือ recovery=d/(1−d); เป็นเลขคณิต ไม่ใช่คำทำนายการฟื้นตัว


In [9]:
for loss in [0.20, 0.50, 0.60]:
    trough = 100 * (1 - loss)
    equal_percentage_rebound = trough * (1 + loss)
    required_recovery = loss / (1 - loss)
    assert math.isclose(trough * (1 + required_recovery), 100)
    print(f"Loss {loss:.0%}: trough {trough:.2f}; same-percent rebound {equal_percentage_rebound:.2f}; required gain {required_recovery:.0%}")


Loss 20%: trough 80.00; same-percent rebound 96.00; required gain 25%
Loss 50%: trough 50.00; same-percent rebound 75.00; required gain 100%
Loss 60%: trough 40.00; same-percent rebound 64.00; required gain 150%


## อ่านผลอย่างไร

- Total return = final equity / initial equity − 1
- CAGR = (final equity / initial equity)^(252/n) − 1; n คือจำนวนช่วงผลตอบแทน 259
- Annualized volatility = sample standard deviation ของ net simple returns × √252
- Annualized Sharpe = mean(net simple returns) / sample standard deviation × √252 โดย rf=0
- Drawdown = equity / running peak − 1; MDD คือค่าต่ำสุด (แสดงเป็นค่าติดลบ)
- การ annualize ด้วย √252 เป็น convention ของแบบฝึกหัด ไม่ได้แก้ serial dependence หรือยืนยันความแม่นยำทางสถิติ

การได้ผลดีในข้อมูลสมมติไม่ใช่หลักฐานว่า SMA หรือ volatility targeting ทำกำไรในตลาดจริง
ต้นทุนจริงยังอาจรวม bid–ask spread, slippage, market impact และข้อจำกัด execution ที่แบบฝึกหัดนี้ไม่ได้จำลอง

แหล่งที่ Nuth ให้: Yves Hilpisch, *Python for Algorithmic Trading*, PDF หน้า 84 (lag สัญญาณ),
94 (proportional costs), 100 (data snooping/overfitting), 148–152 (event-based accounting),
278–279 (Sharpe และ relative drawdown); และ transcript *Introduction to Backtesting* ของ Hudson & Thames
สำหรับการแยก side/size และ volatility targeting
ข้อมูลสมมติ สูตรบัญชี post-fee target weight และโค้ดชุดนี้สร้างขึ้นเพื่ออธิบายกลไก ไม่ใช่ผลทดลองตลาดจากแหล่งอ้างอิง
